# Silver Layer: CRM Product Info

**Source**: `databricks_bootcamp_dwb.bronze.crm_prd_info`  
**Target**: `databricks_bootcamp_dwb.silver.crm_products`  
**Architecture**: Medallion (Bronze → Silver)

## Data Quality Analysis Summary

### Identified Issues:
1. **Duplicates**: Same product key with different time periods (temporal dimension) - **KEEP ALL** as they represent price/cost changes over time
2. **String Issues**: 
   - Trailing spaces in `prd_line` column
   - Abbreviations need normalization: R → Road, S → Sport, M → Mountain, T → Touring
3. **Null Values**: Some products missing cost information (valid - products not yet priced)
4. **Date Validation**: 
   - Null `prd_end_dt` is valid (ongoing products)
   - Some records have `prd_end_dt` < `prd_start_dt` (data quality issue)
5. **Column Naming**: Abbreviated column names need to be user-friendly

### Transformation Plan:
1. Read bronze data
2. Analyze data quality (duplicates, dates, nulls)
3. **Preserve all records** - duplicates represent temporal changes
4. Handle date inconsistencies (explore and decide)
5. Normalize string values (trim spaces, expand abbreviations)
6. Rename columns for readability
7. Validate and enforce data types
8. Write to silver table

### Cross-Check Notes (for later):
- ⚠️ Validate product costs against other tables once silver layer is complete
- ⚠️ Validate product line mappings against `databricks_bootcamp_dwb.bronze.erp_px_cat_g1v2`

---

## Data Quality Analysis
Detailed analysis of data quality issues in the bronze table.

In [0]:
SELECT * FROM databricks_bootcamp_dwb.bronze.crm_prd_info
LIMIT 100

In [0]:
-- Check for duplicate product IDs and keys
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT prd_id) as unique_prd_ids,
  COUNT(DISTINCT prd_key) as unique_prd_keys,
  COUNT(*) - COUNT(DISTINCT prd_id) as duplicate_ids,
  COUNT(*) - COUNT(DISTINCT prd_key) as duplicate_keys
FROM databricks_bootcamp_dwb.bronze.crm_prd_info

In [0]:
-- Find product keys with multiple records (temporal dimension)
SELECT 
  prd_key,
  COUNT(*) as record_count,
  MIN(prd_start_dt) as earliest_start,
  MAX(prd_start_dt) as latest_start
FROM databricks_bootcamp_dwb.bronze.crm_prd_info
GROUP BY prd_key
HAVING COUNT(*) > 1
ORDER BY record_count DESC
LIMIT 10

In [0]:
WITH dup_keys AS (
    SELECT 
        prd_key
    FROM databricks_bootcamp_dwb.bronze.crm_prd_info
    GROUP BY prd_key
    HAVING COUNT(*) > 1
)
SELECT *
FROM databricks_bootcamp_dwb.bronze.crm_prd_info
WHERE prd_key IN (SELECT prd_key FROM dup_keys)
-- Find product keys with multiple records (temporal dimension)


In [0]:
-- Check for leading/trailing spaces in string columns
SELECT 
  'prd_line' as column_name,
  prd_line as raw_value,
  CONCAT('"', prd_line, '"') as with_quotes,
  LENGTH(prd_line) as length,
  LENGTH(TRIM(prd_line)) as trimmed_length
FROM databricks_bootcamp_dwb.bronze.crm_prd_info
WHERE prd_line IS NOT NULL
GROUP BY prd_line
ORDER BY prd_line

In [0]:
-- Check distinct values in prd_line for abbreviation mapping
SELECT 
  TRIM(prd_line) as product_line,
  COUNT(*) as count
FROM databricks_bootcamp_dwb.bronze.crm_prd_info
WHERE prd_line IS NOT NULL
GROUP BY TRIM(prd_line)
ORDER BY count DESC

In [0]:
-- Check cost column for nulls and distribution
SELECT 
  COUNT(*) as total_records,
  COUNT(prd_cost) as non_null_cost,
  COUNT(*) - COUNT(prd_cost) as null_cost,
  MIN(prd_cost) as min_cost,
  MAX(prd_cost) as max_cost,
  ROUND(AVG(prd_cost), 2) as avg_cost
FROM databricks_bootcamp_dwb.bronze.crm_prd_info

In [0]:
-- Show products with null cost
SELECT *
FROM databricks_bootcamp_dwb.bronze.crm_prd_info
WHERE prd_cost IS NULL
LIMIT 20

In [0]:
-- Check date columns
SELECT 
  COUNT(*) as total_records,
  COUNT(prd_start_dt) as non_null_start_dates,
  COUNT(prd_end_dt) as non_null_end_dates,
  MIN(prd_start_dt) as earliest_start,
  MAX(prd_start_dt) as latest_start,
  MIN(prd_end_dt) as earliest_end,
  MAX(prd_end_dt) as latest_end
FROM databricks_bootcamp_dwb.bronze.crm_prd_info

In [0]:
-- Check product key format and patterns
SELECT 
  prd_key,
  LENGTH(prd_key) as key_length,
  prd_nm
FROM databricks_bootcamp_dwb.bronze.crm_prd_info
GROUP BY prd_key, prd_nm
ORDER BY key_length DESC
LIMIT 10

---
## Section 1: Read Bronze Data
Load the bronze table into a DataFrame for transformation.

In [0]:
%python
# Read bronze table into DataFrame
df = spark.table("databricks_bootcamp_dwb.bronze.crm_prd_info")

print(f"Total records: {df.count()}")
print("\nSchema:")
df.printSchema()
display(df)

### Explore Date Inconsistencies
**Issue**: Some records have `prd_end_dt` < `prd_start_dt`, which is logically incorrect.  
**Valid Pattern**: Null `prd_end_dt` represents ongoing products (expected behavior).

In [0]:
%python
from pyspark.sql.functions import col

# Find records where end date is before start date
date_issues = df.filter(
    (col("prd_end_dt").isNotNull()) & 
    (col("prd_end_dt") < col("prd_start_dt"))
)

print(f"Records with end_date < start_date: {date_issues.count()}")
print(f"Total records: {df.count()}")
print(f"Percentage affected: {(date_issues.count() / df.count()) * 100:.2f}%")

display(date_issues.orderBy("prd_key"))

In [0]:
%python
# Check null end dates (these are valid - ongoing products)
null_end_dates = df.filter(col("prd_end_dt").isNull())

print(f"Records with null end_date (ongoing products): {null_end_dates.count()}")
print(f"Percentage: {(null_end_dates.count() / df.count()) * 100:.2f}%")

print("\nSample of ongoing products:")
display(null_end_dates.limit(10))

---
## Date Inconsistency Resolution Options

**Problem**: Records where `prd_end_dt` < `prd_start_dt` are logically invalid.

### Option 1: Set Invalid End Dates to NULL
**Rationale**: Treat these as data entry errors; products are likely ongoing.  
**Action**: Set `prd_end_dt = NULL` where `prd_end_dt < prd_start_dt`  
**Impact**: Preserves all records, marks problematic products as ongoing  
**Risk**: Low - maintains data lineage and flags issue

### Option 2: Swap Start and End Dates
**Rationale**: Assume dates were entered in wrong order.  
**Action**: Swap values where `prd_end_dt < prd_start_dt`  
**Impact**: Corrects potential entry errors  
**Risk**: Medium - assumption may not be valid for all cases

### Option 3: Drop Records with Invalid Dates
**Rationale**: Data quality is too poor; exclude from silver layer.  
**Action**: Filter out records where `prd_end_dt < prd_start_dt`  
**Impact**: Removes problematic data  
**Risk**: High - loses potentially valuable records

### Option 4: Keep As-Is and Flag
**Rationale**: Preserve raw data; add flag column for downstream validation.  
**Action**: Add `date_quality_flag` column to mark inconsistent records  
**Impact**: Maintains full data lineage with quality indicator  
**Risk**: Low - allows downstream systems to decide handling

---
**DECISION**: Hybrid approach combining Option 1 and Option 2:
- Rows with **null `prd_end_dt`**: Keep as-is (ongoing products - correct)
- Rows with **`prd_end_dt` < `prd_start_dt`**: Swap start and end dates

⚠️ **Risk Documentation**: The swap assumes dates were entered in the wrong order. This may not be valid for all affected records. Date quality should be validated against source systems or business stakeholders.

In [0]:
%python
from pyspark.sql.functions import col, when

# Swap start and end dates where end_date < start_date (but keep nulls as-is)
df = df.withColumn(
    "prd_start_dt_fixed",
    when(
        (col("prd_end_dt").isNotNull()) & (col("prd_end_dt") < col("prd_start_dt")),
        col("prd_end_dt")  # Swap: use end_dt as new start
    ).otherwise(col("prd_start_dt"))  # Keep original start
).withColumn(
    "prd_end_dt_fixed",
    when(
        (col("prd_end_dt").isNotNull()) & (col("prd_end_dt") < col("prd_start_dt")),
        col("prd_start_dt")  # Swap: use start_dt as new end
    ).otherwise(col("prd_end_dt"))  # Keep original end (including nulls)
)

# Replace original date columns with fixed versions
df = df.drop("prd_start_dt", "prd_end_dt") \
    .withColumnRenamed("prd_start_dt_fixed", "prd_start_dt") \
    .withColumnRenamed("prd_end_dt_fixed", "prd_end_dt")

print("✅ Date inconsistencies resolved: swapped dates where end < start")
print(f"Total records: {df.count()}")

# Verify the fix
remaining_issues = df.filter(
    (col("prd_end_dt").isNotNull()) & 
    (col("prd_end_dt") < col("prd_start_dt"))
).count()
print(f"Remaining date inconsistencies: {remaining_issues}")

---
## Section 2: Data Transformations
Apply transformations step by step to clean and prepare data for silver layer.

### Handle Null Values
Check for and handle null values in critical columns.

In [0]:
%python
from pyspark.sql.functions import sum, col

# Count nulls in each column
null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])

print("Null counts per column:")
display(null_counts)

In [0]:
%python
# Show products with null cost
null_cost_rows = df.filter(col("prd_cost").isNull())

print(f"Products with null cost: {null_cost_rows.count()}")
display(null_cost_rows)

**Decision**: Keep products with null cost as they represent valid products whose cost data is not yet available.  
Null costs will be preserved in the silver layer for downstream handling.

⚠️ **Cross-check Note**: Validate product costs against other tables once silver layer is complete.

### Normalize String Values
Trim whitespace and expand abbreviations in product line.

In [0]:
%python
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Trim whitespace from all string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

print("String columns trimmed")
display(df)

In [0]:
%python
from pyspark.sql.functions import when

# Expand product line abbreviations: R -> Road, S -> Sport, M -> Mountain, T -> Touring
# Note: Cross-check mappings with databricks_bootcamp_dwb.bronze.erp_px_cat_g1v2 once silver layer is complete
df_normalized = df.withColumn(
    "prd_line",
    when(col("prd_line") == "R", "Road")
    .when(col("prd_line") == "S", "Sport")
    .when(col("prd_line") == "M", "Mountain")
    .when(col("prd_line") == "T", "Touring")
    .otherwise(col("prd_line"))
)

print("Product line abbreviations expanded")
display(df_normalized.select("prd_line").distinct())

### Rename Columns
Rename abbreviated column names to user-friendly names.

In [0]:
%python
# Rename columns to more readable names
df_renamed = df_normalized \
    .withColumnRenamed("prd_id", "product_id") \
    .withColumnRenamed("prd_key", "product_key") \
    .withColumnRenamed("prd_nm", "product_name") \
    .withColumnRenamed("prd_cost", "product_cost") \
    .withColumnRenamed("prd_line", "product_line") \
    .withColumnRenamed("prd_start_dt", "start_date") \
    .withColumnRenamed("prd_end_dt", "end_date")

print("Columns renamed")
df_renamed.printSchema()
display(df_renamed)

### Enforce Data Types
Ensure all columns have the correct data types for the silver layer.

In [0]:
%python
from pyspark.sql.types import IntegerType, FloatType, DateType, StringType
from pyspark.sql.functions import col

# Define target data types for each column
type_mappings = {
    "product_id": IntegerType(),
    "product_cost": FloatType(),  # Float to support decimals and aggregations (mean, variance)
    "start_date": DateType(),
    "end_date": DateType(),
    "tmp_index": None  # Keep tmp_index as-is (will drop later)
}

# Apply type casting
for field in df_renamed.schema.fields:
    column_name = field.name
    
    if column_name in type_mappings:
        target_type = type_mappings[column_name]
        if target_type is not None:
            df_renamed = df_renamed.withColumn(column_name, col(column_name).cast(target_type))
    else:
        # All other columns should be StringType
        df_renamed = df_renamed.withColumn(column_name, col(column_name).cast(StringType()))

print("Data types after enforcement:")
df_renamed.printSchema()

In [0]:
%python
# Drop temporary index column
df_clean = df_renamed.drop("tmp_index")

print(f"Final record count: {df_clean.count()}")
df_clean.printSchema()
display(df_clean)

---
## Sanity Checks
Validate the final DataFrame before writing to silver layer.

In [0]:
%python
from pyspark.sql.functions import col, count

print("=== Final Data Quality Checks ===")
print(f"\nTotal records: {df_clean.count()}")
print(f"Unique product_ids: {df_clean.select('product_id').distinct().count()}")
print(f"Unique product_keys: {df_clean.select('product_key').distinct().count()}")

# Check duplicate product_keys (temporal records - expected)
dup_check = df_clean.groupBy("product_key").agg(count("*").alias("count")).filter(col("count") > 1)
print(f"\nDuplicate product_keys (temporal changes): {dup_check.count()}")

# Verify no trailing spaces
print("\nDistinct product lines:")
display(df_clean.select("product_line").distinct())

# Check null distribution
null_summary = df_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns])
print("\nNull counts:")
display(null_summary)

In [0]:
%python
# Show sample of cleaned data
print("Sample of final cleaned data:")
display(df_clean.limit(20))

---
## Write to Silver Layer
Write the cleaned and transformed data to the silver table.

In [0]:
%python
# Write to silver table with schema overwrite
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_bootcamp_dwb.silver.crm_products")

print("✅ Data successfully written to databricks_bootcamp_dwb.silver.crm_products")

In [0]:
-- Verify the silver table
SELECT 
  COUNT(*) as total_products,
  COUNT(DISTINCT product_key) as unique_products,
  COUNT(product_cost) as products_with_cost,
  MIN(start_date) as earliest_date,
  MAX(start_date) as latest_date
FROM databricks_bootcamp_dwb.silver.crm_products

In [0]:
SELECT * FROM databricks_bootcamp_dwb.silver.crm_products
LIMIT 5

---
## Summary

### Transformations Applied:
✅ **Duplicates Preserved**: All records kept - duplicates represent price/cost changes over time (temporal dimension)  
✅ **Date Inconsistency Resolution**: Hybrid approach applied:
   - Rows with null `end_date`: Kept as-is (ongoing products)
   - Rows with `end_date < start_date`: Start and end dates swapped
   - ⚠️ **Risk**: Assumes dates were entered in wrong order; may not be valid for all records  
✅ **String Normalization**: Trimmed whitespace, expanded abbreviations (R→Road, S→Sport, M→Mountain, T→Touring)  
✅ **Column Renaming**: Applied user-friendly column names  
✅ **Data Types Enforced**: All columns have correct types (product_cost as Float for aggregations)  
✅ **Null Handling**: Preserved null costs for valid products missing cost data  

### Silver Table:
**Table**: `databricks_bootcamp_dwb.silver.crm_products`  
**Columns**: product_id, product_key, product_name, product_cost (Float), product_line, start_date, end_date  
**Temporal Dimension**: Multiple records per product_key capture price changes over time  
**Ready for**: Gold layer aggregations and analytics

### Cross-Check Items (Post Silver Layer):
⚠️ **Product Costs**: Validate against other tables once silver layer is complete  
⚠️ **Product Line Mappings**: Validate against `databricks_bootcamp_dwb.bronze.erp_px_cat_g1v2` once silver layer is complete  
⚠️ **Date Quality**: Validate swapped start/end dates against source systems or business stakeholders (200 records affected)